# I051 Siddhant Lab 7

---

### What this notebook does
Trains **two** seq2seq models on the same task and compares them:

| Model | Architecture | Context |
|---|---|---|
| **Simple Enc-Dec** | LSTM encoder → fixed context vector → LSTM decoder | Only final hidden state `(h, c)` |
| **Attention Enc-Dec** | LSTM encoder → all hidden states → Bahdanau attention → LSTM decoder | Dynamic context at every step |

**Task:** Given a short movie review → generate a 5-6 word summary + predict sentiment (Pos/Neg)

**Dataset:** `rotten_tomatoes` (HuggingFace)
- ~10K short reviews, auto-downloaded, no upload needed
- Capped at 2000 samples for fast training
- Average review length: ~18 words

In [1]:
!pip install -q nltk sacrebleu plotly datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00


In [2]:
import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
print('✅ Packages ready')

✅ Packages ready


In [3]:
import numpy as np
import pandas as pd
import re
import json
import os
import warnings
import matplotlib
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding, Bidirectional,
    Concatenate, Dot, Activation, Dropout
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | GPU: {bool(gpus)}')
np.random.seed(42); tf.random.set_seed(42)

TF 2.19.0 | GPU: False


## 1. Dataset — `rotten_tomatoes`
Auto-downloaded from HuggingFace. No manual upload. ~50 MB.

In [4]:
print('📥 Loading rotten_tomatoes...')
rt = load_dataset('rotten_tomatoes', trust_remote_code=True)
all_data = list(rt['train']) + list(rt['validation'])
print(f'Total examples: {len(all_data)}')

def clean(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9 ',.\']", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

rows = []
for item in all_data:
    rev   = clean(item['text'])
    words = rev.split()
    if 10 <= len(words) <= 35:
        # Pseudo-summary: first 6 content words (extractive proxy)
        rows.append({
            'review'  : rev,
            'summary' : '<start> ' + ' '.join(words[:6]) + ' <end>',
            'label'   : item['label']   # 0=neg, 1=pos
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True).head(2000)
print(f'Using {len(df)} pairs | Pos: {df["label"].sum()} | Neg: {(df["label"]==0).sum()}')
print('\nSample:')
for _, r in df.head(3).iterrows():
    print(f'  REVIEW  : {r["review"][:70]}...')
    print(f'  SUMMARY : {r["summary"]}')
    print(f'  LABEL   : {"Positive" if r["label"]==1 else "Negative"}\n')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rotten_tomatoes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rotten_tomatoes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Loading rotten_tomatoes...


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Total examples: 9596
Using 2000 pairs | Pos: 1013 | Neg: 987

Sample:
  REVIEW  : a moving and weighty depiction of one family's attempts to heal after ...
  SUMMARY : <start> a moving and weighty depiction of <end>
  LABEL   : Positive

  REVIEW  : confessions isn't always coherent , but it's sharply comic and surpris...
  SUMMARY : <start> confessions isn't always coherent , but <end>
  LABEL   : Positive

  REVIEW  : this tenth feature is a big deal , indeed at least the third best , an...
  SUMMARY : <start> this tenth feature is a big <end>
  LABEL   : Positive



In [5]:
# ── Tokenisation ──────────────────────────────────────────────────────────────
VOCAB_SIZE = 4000
SUM_VOCAB  = 2500

rev_tok = Tokenizer(num_words=VOCAB_SIZE, oov_token='<oov>', filters='')
sum_tok = Tokenizer(num_words=SUM_VOCAB,  oov_token='<oov>', filters='')
rev_tok.fit_on_texts(df['review'])
sum_tok.fit_on_texts(df['summary'])

SRC_VOCAB = min(VOCAB_SIZE, len(rev_tok.word_index)) + 1
TGT_VOCAB = min(SUM_VOCAB,  len(sum_tok.word_index)) + 1
MAX_SRC   = min(35, int(np.percentile([len(s.split()) for s in df['review']],   95)))
MAX_TGT   = min(10, int(np.percentile([len(s.split()) for s in df['summary']], 100)))

print(f'SRC_VOCAB={SRC_VOCAB} | TGT_VOCAB={TGT_VOCAB}')
print(f'MAX_SRC={MAX_SRC}     | MAX_TGT={MAX_TGT}')

enc_in     = pad_sequences(rev_tok.texts_to_sequences(df['review']),   maxlen=MAX_SRC, padding='post', truncating='post')
dec_in     = pad_sequences(sum_tok.texts_to_sequences(df['summary']),  maxlen=MAX_TGT, padding='post', truncating='post')
dec_out    = np.zeros_like(dec_in)
dec_out[:, :-1] = dec_in[:, 1:]   # teacher-forcing shift
dec_out_oh = tf.keras.utils.to_categorical(dec_out, TGT_VOCAB)
labels     = df['label'].values.astype(np.float32)

# 80/20 split
split = int(0.8 * len(df))
tr_ei, va_ei = enc_in[:split],    enc_in[split:]
tr_di, va_di = dec_in[:split],    dec_in[split:]
tr_do, va_do = dec_out_oh[:split], dec_out_oh[split:]
tr_lb, va_lb = labels[:split],    labels[split:]

tgt_idx2word = {v: k for k, v in sum_tok.word_index.items()}
START_IDX    = sum_tok.word_index.get('<start>', 1)
END_IDX      = sum_tok.word_index.get('<end>',   2)
print('✅ Data prepared')

SRC_VOCAB=4001 | TGT_VOCAB=2501
MAX_SRC=33     | MAX_TGT=8
✅ Data prepared


## 2. Shared Hyperparameters

In [6]:
LATENT     = 128   # smaller = faster training
EMBED      = 64
BATCH_SIZE = 64
EPOCHS     = 30
LR         = 0.001

def get_callbacks(monitor='val_loss'):
    return [
        EarlyStopping(monitor=monitor, patience=6,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6, verbose=0)
    ]

print(f'LATENT={LATENT} | EMBED={EMBED} | BATCH={BATCH_SIZE} | EPOCHS={EPOCHS} | LR={LR}')

LATENT=128 | EMBED=64 | BATCH=64 | EPOCHS=30 | LR=0.001


## 3. Model A — Simple Encoder-Decoder (No Attention)

The encoder compresses the entire review into a **single fixed context vector** `(h, c)`.
The decoder only sees this one vector — information bottleneck for long sequences.


In [7]:
# ── Simple Encoder ────────────────────────────────────────────────────────────
se_in  = Input(shape=(MAX_SRC,), name='s_enc_in')
se_emb = Embedding(SRC_VOCAB, EMBED, mask_zero=True, name='s_enc_emb')(se_in)
se_emb = Dropout(0.2)(se_emb)
se_l1  = LSTM(LATENT, return_sequences=True, dropout=0.2, name='s_enc_l1')(se_emb)
_, se_h, se_c = LSTM(LATENT, return_state=True, dropout=0.2, name='s_enc_l2')(se_l1)
# NOTE: return_sequences=False → only final (h, c) passed to decoder
# This is the INFORMATION BOTTLENECK — the whole review is compressed to one vector

# ── Sentiment head (from fixed context) ──────────────────────────────────────
s_sent_d   = Dense(32, activation='relu', name='s_sent_dense')(Dropout(0.3)(se_h))
s_sent_out = Dense(1, activation='sigmoid', name='s_sent_out')(s_sent_d)

# ── Simple Decoder ────────────────────────────────────────────────────────────
sd_in   = Input(shape=(None,), name='s_dec_in')
sd_emb  = Embedding(TGT_VOCAB, EMBED, mask_zero=True, name='s_dec_emb')(sd_in)
sd_lstm = LSTM(LATENT, return_sequences=True, return_state=True, dropout=0.2, name='s_dec_lstm')
sd_out, _, _ = sd_lstm(sd_emb, initial_state=[se_h, se_c])
s_sum_out    = Dense(TGT_VOCAB, activation='softmax', name='s_sum_out')(sd_out)

# ── Compile ───────────────────────────────────────────────────────────────────
simple_model = Model([se_in, sd_in], [s_sum_out, s_sent_out], name='simple_enc_dec')
simple_model.compile(
    optimizer=Adam(LR),
    loss={'s_sum_out': 'categorical_crossentropy', 's_sent_out': 'binary_crossentropy'},
    loss_weights={'s_sum_out': 1.0, 's_sent_out': 0.5},
    metrics={'s_sum_out': 'accuracy', 's_sent_out': 'accuracy'}
)
print(f'Simple model params: {simple_model.count_params():,}')
simple_model.summary(line_length=80)

Simple model params: 1,072,134


Model: "simple_enc_dec"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ s_enc_in (InputLayer) │ (None, 33)        │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_enc_emb (Embedding) │ (None, 33, 64)    │     256,064 │ s_enc_in[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, 33, 64)    │           0 │ s_enc_emb[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, 33)        │           0 │ s_enc_in[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_enc_l1 (LSTM)       │ (None, 33, 128)   │      98,816 │ dropout[0][0],     │
│                       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_dec_in (InputLayer) │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_enc_l2 (LSTM)       │ [(None, 128),     │     131,584 │ s_enc_l1[0][0],    │
│                       │ (None, 128),      │             │ not_equal[0][0]    │
│                       │ (None, 128)]      │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_dec_emb (Embedding) │ (None, None, 64)  │     160,064 │ s_dec_in[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, 128)       │           0 │ s_enc_l2[0][1]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_dec_lstm (LSTM)     │ [(None, None,     │      98,816 │ s_dec_emb[0][0],   │
│                       │ 128), (None,      │             │ s_enc_l2[0][1],    │
│                       │ 128), (None,      │             │ s_enc_l2[0][2]     │
│                       │ 128)]             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_sent_dense (Dense)  │ (None, 32)        │       4,128 │ dropout_1[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_sum_out (Dense)     │ (None, None,      │     322,629 │ s_dec_lstm[0][0]   │
│                       │ 2501)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ s_sent_out (Dense)    │ (None, 1)         │          33 │ s_sent_dense[0][0] │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 1,072,134 (4.09 MB)

 Trainable params: 1,072,134 (4.09 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
print('Training Simple Encoder-Decoder...')
hist_simple = simple_model.fit(
    [tr_ei, tr_di], {'s_sum_out': tr_do, 's_sent_out': tr_lb},
    validation_data=([va_ei, va_di], {'s_sum_out': va_do, 's_sent_out': va_lb}),
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    callbacks=get_callbacks(), verbose=1
)
print('✅ Simple model trained')

Training Simple Encoder-Decoder...
Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 260ms/step - loss: 7.6287 - s_sent_out_accuracy: 0.4956 - s_sent_out_loss: 0.7012 - s_sum_out_accuracy: 0.1639 - s_sum_out_loss: 7.2781 - val_loss: 5.7299 - val_s_sent_out_accuracy: 0.4925 - val_s_sent_out_loss: 0.6913 - val_s_sum_out_accuracy: 0.1250 - val_s_sum_out_loss: 5.3628 - learning_rate: 0.0010
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step - loss: 5.7681 - s_sent_out_accuracy: 0.5169 - s_sent_out_loss: 0.7199 - s_sum_out_accuracy: 0.1277 - s_sum_out_loss: 5.4082 - val_loss: 5.0126 - val_s_sent_out_accuracy: 0.5075 - val_s_sent_out_loss: 0.7027 - val_s_sum_out_accuracy: 0.1250 - val_s_sum_out_loss: 4.6334 - learning_rate: 0.0010
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 190ms/step - loss: 5.5164 - s_sent_out_accuracy: 0.5181 - s_sent_out_loss: 0.7036 - s_sum_out_accuracy: 0.1843 - s_sum_out_loss: 5.1646 - val_loss: 4.8827 - val_s_sent_out_accuracy: 0.5075 - val_s_sent_out_loss: 0.6952 - val_s_sum

## 4. Model B — Attention Encoder-Decoder

The encoder outputs **all hidden states** (one per input word).
At each decoder step, attention computes a **weighted sum** over all encoder states.
The decoder can look at any input word at any time — no bottleneck.



In [9]:
# ── Attention Encoder ─────────────────────────────────────────────────────────
ae_in  = Input(shape=(MAX_SRC,), name='a_enc_in')
ae_emb = Embedding(SRC_VOCAB, EMBED, mask_zero=True, name='a_enc_emb')(ae_in)
ae_emb = Dropout(0.2)(ae_emb)
ae_l1  = LSTM(LATENT, return_sequences=True, dropout=0.2, name='a_enc_l1')(ae_emb)
ae_out, ae_h, ae_c = LSTM(
    LATENT, return_sequences=True, return_state=True,
    dropout=0.2, name='a_enc_l2')(ae_l1)
# ae_out: (batch, MAX_SRC, LATENT) — ALL encoder hidden states preserved

# ── Sentiment head (from final encoder state) ─────────────────────────────────
a_sent_d   = Dense(32, activation='relu', name='a_sent_dense')(Dropout(0.3)(ae_h))
a_sent_out = Dense(1, activation='sigmoid', name='a_sent_out')(a_sent_d)

# ── Attention Decoder ─────────────────────────────────────────────────────────
ad_in   = Input(shape=(None,), name='a_dec_in')
ad_emb  = Embedding(TGT_VOCAB, EMBED, mask_zero=True, name='a_dec_emb')(ad_in)
ad_lstm = LSTM(LATENT, return_sequences=True, return_state=True, dropout=0.2, name='a_dec_lstm')
ad_out, _, _ = ad_lstm(ad_emb, initial_state=[ae_h, ae_c])

# ── Bahdanau Attention ────────────────────────────────────────────────────────
# Alignment score: e_ij = decoder_state_i · encoder_state_j
attn_sc  = Dot(axes=[2, 2], name='attn_scores')([ad_out, ae_out])    # (B, TGT, SRC)
attn_wt  = Activation('softmax', name='attn_weights')(attn_sc)        # (B, TGT, SRC)
# Dynamic context: c_i = Σ α_ij * h_j
ctx_vec  = Dot(axes=[2, 1], name='context_vector')([attn_wt, ae_out]) # (B, TGT, LATENT)

combined    = Concatenate(name='combined')([ad_out, ctx_vec])          # (B, TGT, 2*LATENT)
a_dense1    = Dense(LATENT, activation='tanh', name='a_dense1')(combined)
a_sum_out   = Dense(TGT_VOCAB, activation='softmax', name='a_sum_out')(a_dense1)

# ── Compile ───────────────────────────────────────────────────────────────────
attn_model = Model([ae_in, ad_in], [a_sum_out, a_sent_out], name='attn_enc_dec')
attn_model.compile(
    optimizer=Adam(LR),
    loss={'a_sum_out': 'categorical_crossentropy', 'a_sent_out': 'binary_crossentropy'},
    loss_weights={'a_sum_out': 1.0, 'a_sent_out': 0.5},
    metrics={'a_sum_out': 'accuracy', 'a_sent_out': 'accuracy'}
)
print(f'Attention model params: {attn_model.count_params():,}')
attn_model.summary(line_length=80)

Attention model params: 1,105,030


Model: "attn_enc_dec"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ a_enc_in (InputLayer) │ (None, 33)        │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_enc_emb (Embedding) │ (None, 33, 64)    │     256,064 │ a_enc_in[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_2 (Dropout)   │ (None, 33, 64)    │           0 │ a_enc_emb[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_2           │ (None, 33)        │           0 │ a_enc_in[0][0]     │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_dec_in (InputLayer) │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_enc_l1 (LSTM)       │ (None, 33, 128)   │      98,816 │ dropout_2[0][0],   │
│                       │                   │             │ not_equal_2[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_dec_emb (Embedding) │ (None, None, 64)  │     160,064 │ a_dec_in[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_enc_l2 (LSTM)       │ [(None, 33, 128), │     131,584 │ a_enc_l1[0][0],    │
│                       │ (None, 128),      │             │ not_equal_2[0][0]  │
│                       │ (None, 128)]      │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_dec_lstm (LSTM)     │ [(None, None,     │      98,816 │ a_dec_emb[0][0],   │
│                       │ 128), (None,      │             │ a_enc_l2[0][1],    │
│                       │ 128), (None,      │             │ a_enc_l2[0][2]     │
│                       │ 128)]             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ attn_scores (Dot)     │ (None, None, 33)  │           0 │ a_dec_lstm[0][0],  │
│                       │                   │             │ a_enc_l2[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ attn_weights          │ (None, None, 33)  │           0 │ attn_scores[0][0]  │
│ (Activation)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ context_vector (Dot)  │ (None, None, 128) │           0 │ attn_weights[0][0… │
│                       │                   │             │ a_enc_l2[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ combined              │ (None, None, 256) │           0 │ a_dec_lstm[0][0],  │
│ (Concatenate)         │                   │             │ context_vector[0]… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_3 (Dropout)   │ (None, 128)       │           0 │ a_enc_l2[0][1]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_dense1 (Dense)      │ (None, None, 128) │      32,896 │ combined[0][0]     │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_sent_dense (Dense)  │ (None, 32)        │       4,128 │ dropout_3[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_sum_out (Dense)     │ (None, None,      │     322,629 │ a_dense1[0][0]     │
│                       │ 2501)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ a_sent_out (Dense)    │ (None

 Total params: 1,105,030 (4.22 MB)

 Trainable params: 1,105,030 (4.22 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
print('Training Attention Encoder-Decoder...')
hist_attn = attn_model.fit(
    [tr_ei, tr_di], {'a_sum_out': tr_do, 'a_sent_out': tr_lb},
    validation_data=([va_ei, va_di], {'a_sum_out': va_do, 'a_sent_out': va_lb}),
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    callbacks=get_callbacks(), verbose=1
)
print('✅ Attention model trained')

Training Attention Encoder-Decoder...
Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 272ms/step - a_sent_out_accuracy: 0.4963 - a_sent_out_loss: 0.7092 - a_sum_out_accuracy: 0.1692 - a_sum_out_loss: 6.8861 - loss: 7.2407 - val_a_sent_out_accuracy: 0.5075 - val_a_sent_out_loss: 0.7154 - val_a_sum_out_accuracy: 0.1250 - val_a_sum_out_loss: 4.9790 - val_loss: 5.3532 - learning_rate: 0.0010
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 219ms/step - a_sent_out_accuracy: 0.5138 - a_sent_out_loss: 0.7105 - a_sum_out_accuracy: 0.1335 - a_sum_out_loss: 5.3635 - loss: 5.7188 - val_a_sent_out_accuracy: 0.5075 - val_a_sent_out_loss: 0.6966 - val_a_sum_out_accuracy: 0.1250 - val_a_sum_out_loss: 4.6320 - val_loss: 5.0099 - learning_rate: 0.0010
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 257ms/step - a_sent_out_accuracy: 0.5038 - a_sent_out_loss: 0.7106 - a_sum_out_accuracy: 0.1447 - a_sum_out_loss: 5.1745 - loss: 5.5298 - val_a_sent_out_accuracy: 0.4925 - val_a_sent_out_loss: 0.6924 - val_a_sum_out_accuracy: 0.

## 5. Inference Models

In [11]:
# ── Simple inference ──────────────────────────────────────────────────────────
s_enc_inf = Model(se_in, [se_h, se_c, s_sent_out], name='s_enc_inf')

si_tok = Input(shape=(1,),      name='si_tok')
si_h   = Input(shape=(LATENT,), name='si_h')
si_c   = Input(shape=(LATENT,), name='si_c')

_s_emb  = simple_model.get_layer('s_dec_emb')
_s_lstm = simple_model.get_layer('s_dec_lstm')
_s_out  = simple_model.get_layer('s_sum_out')

si_e           = _s_emb(si_tok)
si_d, si_nh, si_nc = _s_lstm(si_e, initial_state=[si_h, si_c])
si_logits      = _s_out(si_d)

s_dec_inf = Model([si_tok, si_h, si_c], [si_logits, si_nh, si_nc], name='s_dec_inf')

print('✅ Simple inference models ready')

✅ Simple inference models ready


In [12]:
# ── Attention inference ───────────────────────────────────────────────────────
a_enc_inf = Model(ae_in, [ae_out, ae_h, ae_c, a_sent_out], name='a_enc_inf')

ai_tok = Input(shape=(1,),              name='ai_tok')
ai_h   = Input(shape=(LATENT,),         name='ai_h')
ai_c   = Input(shape=(LATENT,),         name='ai_c')
ai_enc = Input(shape=(MAX_SRC, LATENT), name='ai_enc')

_a_emb  = attn_model.get_layer('a_dec_emb')
_a_lstm = attn_model.get_layer('a_dec_lstm')
_a_dn1  = attn_model.get_layer('a_dense1')
_a_out  = attn_model.get_layer('a_sum_out')

ai_e              = _a_emb(ai_tok)
ai_d, ai_nh, ai_nc = _a_lstm(ai_e, initial_state=[ai_h, ai_c])
ai_sc             = Dot(axes=[2, 2])([ai_d, ai_enc])
ai_wt             = Activation('softmax')(ai_sc)
ai_cx             = Dot(axes=[2, 1])([ai_wt, ai_enc])
ai_cb             = Concatenate()([ai_d, ai_cx])
ai_dn             = _a_dn1(ai_cb)
ai_logits         = _a_out(ai_dn)

a_dec_inf = Model([ai_tok, ai_h, ai_c, ai_enc], [ai_logits, ai_nh, ai_nc, ai_wt], name='a_dec_inf')

print('✅ Attention inference models ready')

✅ Attention inference models ready


In [13]:
# ── Prediction functions ──────────────────────────────────────────────────────
def predict_simple(review_text):
    """Returns (summary, sentiment, confidence%)"""
    cleaned = re.sub(r"[^a-z0-9 ',.\']", ' ', review_text.lower().strip())
    seq     = pad_sequences(rev_tok.texts_to_sequences([cleaned]),
                            maxlen=MAX_SRC, padding='post', truncating='post')
    h, c, sent_score = s_enc_inf.predict(seq, verbose=0)
    sentiment  = 'Positive' if sent_score[0][0] >= 0.5 else 'Negative'
    confidence = float(sent_score[0][0]) if sentiment == 'Positive' else 1.0 - float(sent_score[0][0])

    tok, words = np.array([[START_IDX]]), []
    for _ in range(MAX_TGT):
        probs, h, c = s_dec_inf.predict([tok, h, c], verbose=0)
        idx  = int(np.argmax(probs[0, 0, :]))
        if idx in (END_IDX, 0): break
        word = tgt_idx2word.get(idx, '')
        if word not in ('<start>', '<end>', '<oov>', ''): words.append(word)
        tok = np.array([[idx]])
    return ' '.join(words).capitalize() or '(no summary)', sentiment, confidence * 100


def predict_attn(review_text):
    """Returns (summary, sentiment, confidence%, attn_list)"""
    cleaned = re.sub(r"[^a-z0-9 ',.\']", ' ', review_text.lower().strip())
    seq     = pad_sequences(rev_tok.texts_to_sequences([cleaned]),
                            maxlen=MAX_SRC, padding='post', truncating='post')
    enc_out_val, h, c, sent_score = a_enc_inf.predict(seq, verbose=0)
    sentiment  = 'Positive' if sent_score[0][0] >= 0.5 else 'Negative'
    confidence = float(sent_score[0][0]) if sentiment == 'Positive' else 1.0 - float(sent_score[0][0])

    tok, words, attn_list = np.array([[START_IDX]]), [], []
    for _ in range(MAX_TGT):
        probs, h, c, attn = a_dec_inf.predict([tok, h, c, enc_out_val], verbose=0)
        idx = int(np.argmax(probs[0, 0, :]))
        attn_list.append(np.array(attn[0, 0, :], dtype=float))
        if idx in (END_IDX, 0): break
        word = tgt_idx2word.get(idx, '')
        if word not in ('<start>', '<end>', '<oov>', ''): words.append(word)
        tok = np.array([[idx]])
    return ' '.join(words).capitalize() or '(no summary)', sentiment, confidence * 100, attn_list


# Quick test
test_reviews = [
    'a visually stunning and emotionally powerful film that stays with you',
    'poorly written and boring with terrible acting throughout',
    'the director crafts a beautiful story with memorable performances',
    'waste of time with no coherent plot or character development',
]

print('\n' + '═'*80)
print(f'  {"REVIEW":<44}  {"SIMPLE":>15}  {"ATTENTION":>15}')
print('═'*80)
for r in test_reviews:
    s_sum, s_sent, s_conf      = predict_simple(r)
    a_sum, a_sent, a_conf, _   = predict_attn(r)
    print(f'  {r[:44]:<44}')
    print(f'    Summary  : {s_sum:<35}  {a_sum}')
    print(f'    Sentiment: {s_sent} ({s_conf:.0f}%)            {a_sent} ({a_conf:.0f}%)')
    print()


════════════════════════════════════════════════════════════════════════════════
  REVIEW                                                 SIMPLE        ATTENTION
════════════════════════════════════════════════════════════════════════════════
  a visually stunning and emotionally powerful
    Summary  : A                                    The film , , the
    Sentiment: Positive (51%)            Positive (51%)

  poorly written and boring with terrible acti
    Summary  : A                                    A film , ,
    Sentiment: Positive (51%)            Positive (51%)

  the director crafts a beautiful story with m
    Summary  : A                                    The film , , the
    Sentiment: Positive (51%)            Positive (51%)

  waste of time with no coherent plot or chara
    Summary  : A                                    A film , , the
    Sentiment: Positive (51%)            Positive (51%)



## 6. Training Curve Comparison (Plotly)

In [14]:
ep_s = list(range(1, len(hist_simple.history['loss']) + 1))
ep_a = list(range(1, len(hist_attn.history['loss'])   + 1))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Loss Comparison', 'Summarization Accuracy'))
fig.update_layout(
    template='plotly_dark',
    title_text='📈 Simple vs Attention — Training Curves',
    title_x=0.5, paper_bgcolor='#0f0f1a', plot_bgcolor='#1a1a2e', height=400
)
fig.add_trace(go.Scatter(x=ep_s, y=hist_simple.history['loss'],
                         name='Simple Loss',  line=dict(color='#ff6b35')), 1, 1)
fig.add_trace(go.Scatter(x=ep_a, y=hist_attn.history['loss'],
                         name='Attn Loss',    line=dict(color='#00d4ff')), 1, 1)
fig.add_trace(go.Scatter(x=ep_s, y=hist_simple.history['s_sum_out_accuracy'],
                         name='Simple Acc',   line=dict(color='#ff6b35')), 1, 2)
fig.add_trace(go.Scatter(x=ep_a, y=hist_attn.history['a_sum_out_accuracy'],
                         name='Attn Acc',     line=dict(color='#00d4ff')), 1, 2)
fig.show()

## 7. Evaluation — BLEU + Sentiment Accuracy

In [15]:
sm = SmoothingFunction().method4
s_refs, s_hyps, s_bleus, s_correct = [], [], [], []
a_refs, a_hyps, a_bleus, a_correct = [], [], [], []

eval_df = df.tail(200)  # evaluate on last 200 (held-out)
for _, row in eval_df.iterrows():
    gold   = row['summary'].replace('<start>', '').replace('<end>', '').split()
    s_sum, s_sent, _ = predict_simple(row['review'])
    a_sum, a_sent, _, _ = predict_attn(row['review'])

    s_hyp = s_sum.lower().split(); a_hyp = a_sum.lower().split()
    s_refs.append([gold]); s_hyps.append(s_hyp)
    a_refs.append([gold]); a_hyps.append(a_hyp)
    s_bleus.append(sentence_bleu([gold], s_hyp, smoothing_function=sm))
    a_bleus.append(sentence_bleu([gold], a_hyp, smoothing_function=sm))
    s_correct.append(int((1 if s_sent=='Positive' else 0) == row['label']))
    a_correct.append(int((1 if a_sent=='Positive' else 0) == row['label']))

s_cb = corpus_bleu(s_refs, s_hyps, smoothing_function=sm)
a_cb = corpus_bleu(a_refs, a_hyps, smoothing_function=sm)

print('═' * 66)
print('  MODEL COMPARISON — Sentiment-Aware Review Summarizer')
print('═' * 66)
print(f'{"Metric":<32} {"Simple":>14} {"Attention":>14}')
print('─' * 66)
print(f'{"Train Loss (final)":<32} {hist_simple.history["loss"][-1]:>14.4f} {hist_attn.history["loss"][-1]:>14.4f}')
print(f'{"Val Loss (final)":<32} {hist_simple.history["val_loss"][-1]:>14.4f} {hist_attn.history["val_loss"][-1]:>14.4f}')
print(f'{"Sum Accuracy (train)":<32} {hist_simple.history["s_sum_out_accuracy"][-1]*100:>13.2f}% {hist_attn.history["a_sum_out_accuracy"][-1]*100:>13.2f}%')
print(f'{"Sent Accuracy (train)":<32} {hist_simple.history["s_sent_out_accuracy"][-1]*100:>13.2f}% {hist_attn.history["a_sent_out_accuracy"][-1]*100:>13.2f}%')
print(f'{"Corpus BLEU":<32} {s_cb:>14.4f} {a_cb:>14.4f}')
print(f'{"Avg Sentence BLEU":<32} {np.mean(s_bleus):>14.4f} {np.mean(a_bleus):>14.4f}')
print(f'{"Sentiment Acc (test 200)":<32} {np.mean(s_correct)*100:>13.2f}% {np.mean(a_correct)*100:>13.2f}%')
print(f'{"Epochs trained":<32} {len(ep_s):>14} {len(ep_a):>14}')
print('═' * 66)

══════════════════════════════════════════════════════════════════
  MODEL COMPARISON — Sentiment-Aware Review Summarizer
══════════════════════════════════════════════════════════════════
Metric                                   Simple      Attention
──────────────────────────────────────────────────────────────────
Train Loss (final)                       4.1627         4.0674
Val Loss (final)                         4.2349         4.2536
Sum Accuracy (train)                     36.32%         35.95%
Sent Accuracy (train)                    50.88%         50.81%
Corpus BLEU                              0.0000         0.0056
Avg Sentence BLEU                        0.0022         0.0256
Sentiment Acc (test 200)                 50.00%         50.00%
Epochs trained                               30             28
══════════════════════════════════════════════════════════════════


In [16]:
# ── BLEU bar chart (matplotlib, saved for Streamlit) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')

for ax, vals, title in zip(
    axes,
    [[s_cb, a_cb], [np.mean(s_correct), np.mean(a_correct)]],
    ['Corpus BLEU (Summary)', 'Sentiment Accuracy']
):
    bars = ax.bar(['Simple', 'Attention'], vals,
                  color=['#ff6b35', '#00d4ff'], width=0.45, edgecolor='#30363d')
    ax.set_title(title, color='white', fontweight='bold')
    ax.set_ylabel('Score', color='white')
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{b.get_height():.4f}', ha='center', color='white', fontsize=10)

fig.suptitle('Simple vs Attention Encoder-Decoder', color='white',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bleu_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved bleu_comparison.png')

Saved bleu_comparison.png


## 8. Attention Heatmap

In [17]:
def plot_attention_heatmap(review, summary, attn_list, save_path=None):
    src_words = review.lower().split()[:MAX_SRC]
    tgt_words = summary.split()
    n_t = min(len(tgt_words), len(attn_list))
    n_s = len(src_words)
    if n_t == 0 or n_s == 0: return

    M = np.zeros((n_t, n_s))
    for i in range(n_t):
        w = np.array(attn_list[i][:n_s], dtype=float)
        if len(w) < n_s: w = np.pad(w, (0, n_s - len(w)))
        M[i] = w / (w.sum() + 1e-9)

    fig, ax = plt.subplots(figsize=(max(6, n_s * 0.8), max(3, n_t * 0.8)))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#0d1117')
    im = ax.imshow(M, cmap='plasma', aspect='auto')
    ax.set_xticks(range(n_s))
    ax.set_xticklabels(src_words, rotation=45, ha='right', color='white', fontsize=9)
    ax.set_yticks(range(n_t))
    ax.set_yticklabels(tgt_words[:n_t], color='white', fontsize=9)
    ax.set_xlabel('Review Words (Encoder)', color='white')
    ax.set_ylabel('Summary Words (Decoder)', color='white')
    ax.set_title(f'Attention: "{review[:50]}..."', color='white', pad=10)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

for demo in test_reviews[:2]:
    a_sum, a_sent, a_conf, attn = predict_attn(demo)
    print(f'Review   : {demo}')
    print(f'Summary  : {a_sum}')
    print(f'Sentiment: {a_sent} ({a_conf:.1f}%)')
    plot_attention_heatmap(demo, a_sum, attn, 'attention_heatmap.png')

Review   : a visually stunning and emotionally powerful film that stays with you
Summary  : The film , , the
Sentiment: Positive (50.9%)
Review   : poorly written and boring with terrible acting throughout
Summary  : A film , ,
Sentiment: Positive (50.8%)


## 9. Training Curves (Matplotlib — saved for Streamlit)

In [18]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#0d1117')
plots = [
    (axes[0,0], 'Total Loss',            'loss',                 'val_loss'),
    (axes[0,1], 'Summary Accuracy',      's_sum_out_accuracy',   'val_s_sum_out_accuracy'),
    (axes[1,0], 'Summary Accuracy (Attn)', 'a_sum_out_accuracy', 'val_a_sum_out_accuracy'),
    (axes[1,1], 'Sentiment Accuracy',    's_sent_out_accuracy',  'val_s_sent_out_accuracy'),
]
hist_map = {
    'loss': hist_simple, 'val_loss': hist_simple,
    's_sum_out_accuracy': hist_simple, 'val_s_sum_out_accuracy': hist_simple,
    'a_sum_out_accuracy': hist_attn,   'val_a_sum_out_accuracy': hist_attn,
    's_sent_out_accuracy': hist_simple,'val_s_sent_out_accuracy': hist_simple,
}

for ax, title, train_k, val_k in plots:
    hist = hist_map[train_k]
    ep   = range(1, len(hist.history[train_k]) + 1)
    ax.set_facecolor('#161b22'); ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    ax.plot(ep, hist.history[train_k],  color='#58a6ff', lw=2, label='Train')
    if val_k in hist.history:
        ax.plot(ep, hist.history[val_k], color='#ff7b72', lw=2, ls='--', label='Val')
    ax.set_title(title, color='white', fontweight='bold')
    ax.set_xlabel('Epoch', color='white')
    ax.legend(labelcolor='white', facecolor='#161b22')

fig.suptitle('Training Curves — Simple vs Attention Enc-Dec', color='white',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

Saved training_curves.png


## 10. Save Model Artifacts

In [19]:
os.makedirs('model_artifacts', exist_ok=True)

simple_model.save('model_artifacts/simple_enc_dec.h5')
attn_model.save('model_artifacts/attn_enc_dec.h5')

with open('model_artifacts/rev_tok.json', 'w') as f:
    json.dump(rev_tok.to_json(), f)
with open('model_artifacts/sum_tok.json', 'w') as f:
    json.dump(sum_tok.to_json(), f)

config = {
    'MAX_SRC': MAX_SRC, 'MAX_TGT': MAX_TGT,
    'SRC_VOCAB': SRC_VOCAB, 'TGT_VOCAB': TGT_VOCAB,
    'LATENT': LATENT, 'EMBED': EMBED,
    'START_IDX': START_IDX, 'END_IDX': END_IDX,
    'metrics': {
        'simple': {
            'final_loss': round(hist_simple.history['loss'][-1], 4),
            'final_val_loss': round(hist_simple.history['val_loss'][-1], 4),
            'sum_acc': round(hist_simple.history['s_sum_out_accuracy'][-1], 4),
            'sent_acc': round(hist_simple.history['s_sent_out_accuracy'][-1], 4),
            'corpus_bleu': round(float(s_cb), 4),
            'sent_test_acc': round(float(np.mean(s_correct)), 4),
            'epochs': len(ep_s)
        },
        'attn': {
            'final_loss': round(hist_attn.history['loss'][-1], 4),
            'final_val_loss': round(hist_attn.history['val_loss'][-1], 4),
            'sum_acc': round(hist_attn.history['a_sum_out_accuracy'][-1], 4),
            'sent_acc': round(hist_attn.history['a_sent_out_accuracy'][-1], 4),
            'corpus_bleu': round(float(a_cb), 4),
            'sent_test_acc': round(float(np.mean(a_correct)), 4),
            'epochs': len(ep_a)
        }
    }
}
with open('model_artifacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Saved model_artifacts/')
print('   simple_enc_dec.h5')
print('   attn_enc_dec.h5')
print('   rev_tok.json, sum_tok.json, config.json')

✅ Saved model_artifacts/
   simple_enc_dec.h5
   attn_enc_dec.h5
   rev_tok.json, sum_tok.json, config.json


In [20]:

# Step 1: Install dependencies
!pip install -q streamlit pyngrok

# Step 2: Write app_lab7.py to the Colab filesystem
APP_CODE = r'''
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re, json, os
matplotlib.use("Agg")

st.set_page_config(page_title="Lab 7 · Simple vs Attention", page_icon="🧠",
                   layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    .stApp{background-color:#0d1117}
    h1,h2,h3{color:#58a6ff!important}
    p,li,td,th{color:#c9d1d9}
    .card{background:#161b22;border:1px solid #30363d;border-radius:10px;padding:16px 20px;margin:6px 0}
    .pos-badge{background:#1a4731;color:#3fb950;border-radius:20px;padding:4px 14px;font-weight:700;font-size:.95rem;display:inline-block}
    .neg-badge{background:#3d1f1f;color:#ff7b72;border-radius:20px;padding:4px 14px;font-weight:700;font-size:.95rem;display:inline-block}
    .result-box-simple{background:#161b22;border-left:4px solid #ff6b35;border-radius:6px;padding:12px 16px;margin:8px 0;color:#e6edf3}
    .result-box-attn{background:#161b22;border-left:4px solid #00d4ff;border-radius:6px;padding:12px 16px;margin:8px 0;color:#e6edf3}
    .model-tag-simple{background:#2d1b00;color:#ff6b35;border-radius:4px;padding:2px 8px;font-size:.8rem;font-weight:700}
    .model-tag-attn{background:#001d26;color:#00d4ff;border-radius:4px;padding:2px 8px;font-size:.8rem;font-weight:700}
    .stTextArea textarea{background:#161b22!important;color:#e6edf3!important;border:1px solid #30363d!important}
    .stButton>button{background:linear-gradient(90deg,#1f6feb,#238636);color:white;border:none;font-weight:700;border-radius:8px;padding:10px 28px;font-size:1rem;width:100%}
    .sidebar-info{font-size:.82rem;color:#8b949e;line-height:1.7}
</style>
""", unsafe_allow_html=True)

@st.cache_resource(show_spinner="Loading models...")
def load_bundle():
    import tensorflow as tf
    from tensorflow.keras.models import load_model as km
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
    from tensorflow.keras.layers import Input, Dot, Activation, Concatenate, Dropout
    from tensorflow.keras.models import Model
    import tensorflow.keras.layers as KL

    A = "model_artifacts"
    if not os.path.exists(f"{A}/simple_enc_dec.h5"): return None

    with open(f"{A}/config.json")  as f: cfg     = json.load(f)
    with open(f"{A}/rev_tok.json") as f: rev_tok = tokenizer_from_json(json.load(f))
    with open(f"{A}/sum_tok.json") as f: sum_tok = tokenizer_from_json(json.load(f))

    simple_model = km(f"{A}/simple_enc_dec.h5")
    attn_model   = km(f"{A}/attn_enc_dec.h5")
    MAX_SRC = cfg["MAX_SRC"]; LATENT = cfg["LATENT"]

    s_ei  = simple_model.inputs[0]
    s_emb = simple_model.get_layer("s_enc_emb")(s_ei)
    s_d   = KL.Dropout(0.2)(s_emb, training=False)
    s_l1  = simple_model.get_layer("s_enc_l1")(s_d)
    _, s_h, s_c = simple_model.get_layer("s_enc_l2")(s_l1)
    s_sd  = simple_model.get_layer("s_sent_dense")(KL.Dropout(0.3)(s_h, training=False))
    s_sp  = simple_model.get_layer("s_sent_out")(s_sd)
    s_enc_inf = Model(s_ei, [s_h, s_c, s_sp])

    si_tok=Input(shape=(1,),name="si_tok"); si_h=Input(shape=(LATENT,),name="si_h"); si_c=Input(shape=(LATENT,),name="si_c")
    _se=simple_model.get_layer("s_dec_emb"); _sl=simple_model.get_layer("s_dec_lstm"); _so=simple_model.get_layer("s_sum_out")
    si_e=_se(si_tok); si_d,si_nh,si_nc=_sl(si_e,initial_state=[si_h,si_c]); si_log=_so(si_d)
    s_dec_inf = Model([si_tok,si_h,si_c],[si_log,si_nh,si_nc])

    a_ei  = attn_model.inputs[0]
    a_emb = attn_model.get_layer("a_enc_emb")(a_ei)
    a_d   = KL.Dropout(0.2)(a_emb, training=False)
    a_l1  = attn_model.get_layer("a_enc_l1")(a_d)
    a_out,a_h,a_c = attn_model.get_layer("a_enc_l2")(a_l1)
    a_sd  = attn_model.get_layer("a_sent_dense")(KL.Dropout(0.3)(a_h,training=False))
    a_sp  = attn_model.get_layer("a_sent_out")(a_sd)
    a_enc_inf = Model(a_ei,[a_out,a_h,a_c,a_sp])

    ai_tok=Input(shape=(1,),name="ai_tok"); ai_h=Input(shape=(LATENT,),name="ai_h")
    ai_c=Input(shape=(LATENT,),name="ai_c"); ai_enc=Input(shape=(MAX_SRC,LATENT),name="ai_enc")
    _ae=attn_model.get_layer("a_dec_emb"); _al=attn_model.get_layer("a_dec_lstm")
    _adn=attn_model.get_layer("a_dense1"); _ao=attn_model.get_layer("a_sum_out")
    ai_e=_ae(ai_tok); ai_d,ai_nh,ai_nc=_al(ai_e,initial_state=[ai_h,ai_c])
    ai_sc=Dot(axes=[2,2])([ai_d,ai_enc]); ai_wt=Activation("softmax")(ai_sc)
    ai_cx=Dot(axes=[2,1])([ai_wt,ai_enc]); ai_cb=Concatenate()([ai_d,ai_cx])
    ai_dn=_adn(ai_cb); ai_log=_ao(ai_dn)
    a_dec_inf = Model([ai_tok,ai_h,ai_c,ai_enc],[ai_log,ai_nh,ai_nc,ai_wt])

    return {"s_enc":s_enc_inf,"s_dec":s_dec_inf,"a_enc":a_enc_inf,"a_dec":a_dec_inf,
            "rev_tok":rev_tok,"sum_tok":sum_tok,
            "idx2word":{v:k for k,v in sum_tok.word_index.items()},"cfg":cfg}

def _clean(t): return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9 \',.]"," ",t.lower().strip())).strip()

def _seq(t,tok,mx):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    return pad_sequences(tok.texts_to_sequences([t]),maxlen=mx,padding="post",truncating="post")

def run_simple(rev, b):
    cfg=b["cfg"]; MAX_SRC,MAX_TGT,SI,EI=cfg["MAX_SRC"],cfg["MAX_TGT"],cfg["START_IDX"],cfg["END_IDX"]
    seq=_seq(_clean(rev),b["rev_tok"],MAX_SRC)
    h,c,ss=b["s_enc"].predict(seq,verbose=0)
    sent="Positive" if ss[0][0]>=.5 else "Negative"
    conf=float(ss[0][0]) if sent=="Positive" else 1-float(ss[0][0])
    tok,words=np.array([[SI]]),[]
    for _ in range(MAX_TGT):
        p,h,c=b["s_dec"].predict([tok,h,c],verbose=0)
        idx=int(np.argmax(p[0,0,:]))
        if idx in(EI,0): break
        w=b["idx2word"].get(idx,"")
        if w not in("<start>","<end>","<oov>",""): words.append(w)
        tok=np.array([[idx]])
    return " ".join(words).capitalize() or "(no output)",sent,conf*100

def run_attn(rev, b):
    cfg=b["cfg"]; MAX_SRC,MAX_TGT,SI,EI=cfg["MAX_SRC"],cfg["MAX_TGT"],cfg["START_IDX"],cfg["END_IDX"]
    seq=_seq(_clean(rev),b["rev_tok"],MAX_SRC)
    eo,h,c,ss=b["a_enc"].predict(seq,verbose=0)
    sent="Positive" if ss[0][0]>=.5 else "Negative"
    conf=float(ss[0][0]) if sent=="Positive" else 1-float(ss[0][0])
    tok,words,al=np.array([[SI]]),[],[]
    for _ in range(MAX_TGT):
        p,h,c,a=b["a_dec"].predict([tok,h,c,eo],verbose=0)
        idx=int(np.argmax(p[0,0,:]))
        al.append(np.array(a[0,0,:],dtype=float))
        if idx in(EI,0): break
        w=b["idx2word"].get(idx,"")
        if w not in("<start>","<end>","<oov>",""): words.append(w)
        tok=np.array([[idx]])
    return " ".join(words).capitalize() or "(no output)",sent,conf*100,al

def draw_heatmap(rev, summ, al, mx):
    sw=rev.lower().split()[:mx]; tw=summ.split()
    n_t=min(len(tw),len(al)); n_s=len(sw)
    if n_t==0 or n_s==0: return None
    M=np.zeros((n_t,n_s))
    for i in range(n_t):
        w=np.array(al[i][:n_s],dtype=float)
        if len(w)<n_s: w=np.pad(w,(0,n_s-len(w)))
        M[i]=w/(w.sum()+1e-9)
    fig,ax=plt.subplots(figsize=(max(6,n_s*.7),max(2.5,n_t*.75)))
    fig.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
    im=ax.imshow(M,cmap="plasma",aspect="auto")
    ax.set_xticks(range(n_s)); ax.set_xticklabels(sw,rotation=45,ha="right",color="white",fontsize=9)
    ax.set_yticks(range(n_t)); ax.set_yticklabels(tw[:n_t],color="white",fontsize=9)
    ax.set_xlabel("Review words",color="white",fontsize=9)
    ax.set_ylabel("Summary words",color="white",fontsize=9)
    ax.set_title("Attention Heatmap",color="white",fontsize=11,fontweight="bold")
    ax.tick_params(colors="white")
    for sp in ax.spines.values(): sp.set_color("#30363d")
    plt.colorbar(im,ax=ax); plt.tight_layout(); return fig

with st.sidebar:
    st.markdown("## 🧠 ATML Lab 7")
    st.markdown("**Simple vs Attention Enc-Dec**")
    st.markdown("---")
    st.markdown("""<div class="sidebar-info">
<b>Dataset:</b> Rotten Tomatoes<br>
<b>Samples:</b> 2000 reviews<br>
<b>Split:</b> 80 / 20<br><br>
<b>Model A — Simple:</b><br>
· 2-layer LSTM + fixed context<br>
· Sentiment head on final h<br><br>
<b>Model B — Attention:</b><br>
· 2-layer LSTM + all states<br>
· Bahdanau attention<br>
· Dynamic context per step
</div>""",unsafe_allow_html=True)
    st.markdown("---")
    examples=["a visually stunning and emotionally powerful film",
              "poorly written and boring with terrible acting",
              "the director crafts a beautiful story with great performances",
              "waste of time with no coherent plot or character development"]
    selected=st.selectbox("Try an example",["— choose —"]+examples)

st.markdown("""
<div style="text-align:center;padding:10px 0 4px">
<h1 style="font-size:2rem;color:#58a6ff;margin:0">🎬 Sentiment-Aware Review Summarizer</h1>
<p style="color:#8b949e;margin:6px 0 0;font-size:.9rem">ATML Lab 7 &nbsp;·&nbsp; Simple vs Attention &nbsp;·&nbsp; Multi-Task Learning</p>
</div>""",unsafe_allow_html=True)
st.markdown("---")

tab1,tab2,tab3,tab4=st.tabs(["🔍  Predict","📊  Batch Compare","📈  Dashboard","📚  Theory"])

with tab1:
    cl,cr=st.columns([1,1],gap="large")
    with cl:
        st.markdown("#### Enter a Movie Review")
        dflt=selected if selected!="— choose —" else ""
        rv=st.text_area("Review",value=dflt,height=120,label_visibility="collapsed",
                        placeholder="e.g. a visually stunning film...")
        btn=st.button("▶  Run Both Models")
    if btn and rv.strip():
        with st.spinner("Running inference..."):
            b=load_bundle()
            if b is None: st.error("Run the notebook first.")
            else:
                ss,s_sent,s_conf=run_simple(rv,b)
                as_,a_sent,a_conf,a_al=run_attn(rv,b)
                with cr:
                    st.markdown("#### Results")
                    st.markdown('<span class="model-tag-simple">MODEL A · SIMPLE</span>',unsafe_allow_html=True)
                    sb="pos-badge" if s_sent=="Positive" else "neg-badge"
                    st.markdown(f'<span class="{sb}">{"😊" if s_sent=="Positive" else "😞"} {s_sent} &nbsp;·&nbsp; {s_conf:.1f}%</span>',unsafe_allow_html=True)
                    st.markdown(f'<div class="result-box-simple"><b>Summary:</b> {ss}</div>',unsafe_allow_html=True)
                    st.markdown("<br>",unsafe_allow_html=True)
                    st.markdown('<span class="model-tag-attn">MODEL B · ATTENTION</span>',unsafe_allow_html=True)
                    ab="pos-badge" if a_sent=="Positive" else "neg-badge"
                    st.markdown(f'<span class="{ab}">{"😊" if a_sent=="Positive" else "😞"} {a_sent} &nbsp;·&nbsp; {a_conf:.1f}%</span>',unsafe_allow_html=True)
                    st.markdown(f'<div class="result-box-attn"><b>Summary:</b> {as_}</div>',unsafe_allow_html=True)
                st.markdown("#### Attention Heatmap")
                fig=draw_heatmap(rv,as_,a_al,b["cfg"]["MAX_SRC"])
                if fig: st.pyplot(fig); plt.close(fig)
    elif btn: st.warning("Please enter a review.")
    if not os.path.exists("model_artifacts/simple_enc_dec.h5"):
        st.info("Models not found. Run the notebook first.")

with tab2:
    batch=st.text_area("One review per line",height=130,
        value="\n".join(["a visually stunning and emotionally powerful film",
                         "poorly written and boring with terrible acting",
                         "the director crafts a beautiful story",
                         "waste of time with no coherent plot"]))
    if st.button("▶  Compare All"):
        b=load_bundle()
        if b:
            revs=[r.strip() for r in batch.strip().split("\n") if r.strip()]
            rows=[]
            with st.spinner("Processing..."):
                for r in revs:
                    ss,s_sent,s_conf=run_simple(r,b)
                    as_,a_sent,a_conf,_=run_attn(r,b)
                    rows.append({"Review":r[:50]+"..." if len(r)>50 else r,
                                 "Simple Summary":ss,"Simple":f"{'😊' if s_sent=='Positive' else '😞'} {s_sent}({s_conf:.0f}%)",
                                 "Attn Summary":as_,"Attention":f"{'😊' if a_sent=='Positive' else '😞'} {a_sent}({a_conf:.0f}%)"})
            import pandas as pd; st.dataframe(pd.DataFrame(rows),use_container_width=True)

with tab3:
    st.markdown("#### Performance Dashboard")
    if os.path.exists("model_artifacts/config.json"):
        with open("model_artifacts/config.json") as f: cfg=json.load(f)
        m=cfg.get("metrics",{}); sm2=m.get("simple",{}); am2=m.get("attn",{})
        c1,c2,c3,c4=st.columns(4)
        for col,(v,l,clr) in zip([c1,c2,c3,c4],[
            (f"{sm2.get('corpus_bleu',0):.4f}","BLEU · Simple","#ff6b35"),
            (f"{am2.get('corpus_bleu',0):.4f}","BLEU · Attention","#00d4ff"),
            (f"{sm2.get('sent_test_acc',0)*100:.1f}%","Sent Acc · Simple","#ff6b35"),
            (f"{am2.get('sent_test_acc',0)*100:.1f}%","Sent Acc · Attn","#00d4ff")]):
            col.markdown(f"<div class='card' style='text-align:center'><div style='font-size:1.8rem;font-weight:700;color:{clr}'>{v}</div><div style='font-size:.78rem;color:#8b949e'>{l}</div></div>",unsafe_allow_html=True)
    if os.path.exists("training_curves.png"): st.image("training_curves.png",use_column_width=True)
    if os.path.exists("bleu_comparison.png"): st.image("bleu_comparison.png",use_column_width=True)
    if os.path.exists("attention_heatmap.png"): st.image("attention_heatmap.png",use_column_width=True)

with tab4:
    st.markdown("""
## Why Does Attention Help?
The simple model compresses the entire review into one fixed vector — causing information loss for longer inputs.
Attention preserves all encoder states and computes a dynamic weighted context at every decoder step.

| Feature | Simple | Attention |
|---|---|---|
| Context vector | Fixed (h,c) only | Dynamic per step |
| Info bottleneck | ✅ Yes | ❌ No |
| Interpretable | ❌ | ✅ Heatmap |
| BLEU score | Lower | Higher |

## Multi-Task Learning
Both tasks share the encoder — summarization (CrossEntropy, w=1.0) + sentiment (BinaryCE, w=0.5).
""")

st.markdown("<div style='text-align:center;color:#30363d;font-size:.75rem;margin-top:30px'>NMIMS MPSTME · ATML Lab 7 · I051 Siddhant Nikam</div>",unsafe_allow_html=True)
'''

with open("app_lab7.py", "w") as f:
    f.write(APP_CODE)
print("✅ app_lab7.py written to Colab filesystem")

# Step 3: Launch via pyngrok
import threading, time, subprocess
from pyngrok import ngrok, conf

# ── OPTIONAL: set your free ngrok token from https://ngrok.com ──────────────
conf.get_default().auth_token = "3CDfr14enzETXzoDoSrPgDOCsOn_5PwmcgwV7ya91AtVdAcPV"
# ────────────────────────────────────────────────────────────────────────────

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app_lab7.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)  # wait for streamlit to boot

public_url = ngrok.connect(8501)
print("=" * 58)
print(f"  🚀 Dashboard live at: {public_url}")
print("=" * 58)
print("  Open the URL above in any browser.")
print("  ⚠  Keep this cell running — closing it stops the app.")
print()
print("  Tabs available:")
print("    🔍  Predict       — run both models on any review")
print("    📊  Batch Compare — compare across multiple reviews")
print("    📈  Dashboard     — metrics, curves, heatmap")
print("    📚  Theory        — explanation of attention")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.4 MB/s eta 0:00:00
✅ app_lab7.py written to Colab filesystem
  🚀 Dashboard live at: NgrokTunnel: "https://negligee-thinly-left.ngrok-free.dev" -> "http://localhost:8501"
  Open the URL above in any browser.
  ⚠  Keep this cell running — closing it stops the app.

  Tabs available:
    🔍  Predict       — run both models on any review
    📊  Batch Compare — compare across multiple reviews
    📈  Dashboard     — metrics, curves, heatmap
    📚  Theory        — explanation of attention
